# Recommendation / 06 — Select the winner

Select the trained algorithm with highest validation accuracy; macro F1 breaks ties. Do not pick a model from its test score.

Run every cell from top to bottom. Use the project `.venv` kernel in VS Code. No season field is used. Data and test scores are synthetic.

In [1]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml_service/workflows.py").exists())
sys.path.insert(0, str(ROOT))
from ml_service import workflows as w
print("Project:", ROOT)
print("Synthetic demonstration: real-world accuracy has not been measured.")

Project: /workspace/scratch/71e3a62599e6/YatraSense_Organic
Synthetic demonstration: real-world accuracy has not been measured.


In [2]:
print(inspect.getsource(w.select))

def select(task):
    table=pd.read_csv(REPORTS/f'{task}_validation.csv')
    # Respect requested highest validation accuracy; macro F1 breaks ties.
    table=table[table.model!='MajorityBaseline'].sort_values(['accuracy','macro_f1'],ascending=False)
    winner=table.iloc[0].model
    model=joblib.load(MODELS/f'{task}_candidates.joblib')[winner]
    joblib.dump(model,MODELS/f'{task}.joblib')
    choice={'model':winner,'criterion':'highest validation accuracy; macro F1 tie-break','validation':table.iloc[0].to_dict(),'test_used_for_selection':False,'synthetic':True}
    write_json(REPORTS/f'{task}_selection.json',choice)
    return choice



In [3]:
choice = w.select("recommendation")
display(pd.Series(choice))
print("Saved fitted pipeline:", w.MODELS / "recommendation.joblib")

Saved fitted pipeline: /workspace/scratch/71e3a62599e6/YatraSense_Organic/models/recommendation.joblib


model                                                           RandomForest
criterion                    highest validation accuracy; macro F1 tie-break
validation                 {'model': 'RandomForest', 'accuracy': 0.778472...
test_used_for_selection                                                False
synthetic                                                               True
dtype: object

The selected pipeline remains fitted on the training set, preserving the evaluation/inference contract. An 80% target is not guaranteed; results are never rewritten to meet it.